In [5]:
from torch.utils.data import Dataset 
from torch.utils.data import DataLoader
import torch 
import os 
import numpy as np 

train_path: str = "../data/ServerMachineDataset/train"
test_path: str = "../data/ServerMachineDataset/test"
test_label_path: str = "../data/ServerMachineDataset/test_label"


class SMDDataset(Dataset): 

    def __init__(self, root: str, window_size: int = 100): 
        self.root = root 
        self.window_size = window_size
        self.records: list = []
        self.machine_count: int = len(os.listdir(self.root))

        machines: list[str] = os.listdir(self.root)
        for machine_id, machine_filename in enumerate(machines):
            machine_path: str = os.path.join(self.root, machine_filename)

            # read the file 
            machine_data: torch.Tensor = torch.from_numpy(
                np.loadtxt(machine_path, delimiter=",", dtype=np.float32), 
            )

            # split the data into windows
            for i in range(0, len(machine_data) - self.window_size):
                window_data: torch.Tensor = machine_data[i:i+self.window_size]
                y = machine_data[i + self.window_size]
                self.records.append((machine_id, window_data, y))

    def __len__(self) -> int:
        return len(self.records)
    
    def __getitem__(self, idx: int) -> tuple[int, torch.Tensor]:
        return self.records[idx]
    
    
train = SMDDataset(train_path, window_size=100)
test = SMDDataset(test_path, window_size=100)
 


In [6]:
loader = DataLoader(train, batch_size=32, shuffle=True)
machine_ids, sensor, labels = next(iter(loader))
time_window, channels = sensor.shape[1:]
sensor.shape

torch.Size([32, 100, 38])

In [7]:
labels.shape

torch.Size([32, 38])

In [8]:
machine_ids.shape

torch.Size([32])

In [9]:
import torch.nn as nn 

embedding = nn.Embedding(num_embeddings=train.machine_count, embedding_dim=100)
machine_embeds = embedding(machine_ids)

In [10]:
temporal_patterns = 20
kernel_size = 7
stride = 2

conv = nn.Conv1d(
    in_channels=channels, 
    out_channels=temporal_patterns, 
    kernel_size=7, 
    stride=2
)

transformed_window = (time_window - kernel_size) // stride + 1

trf_x = conv(sensor.transpose(1, 2)).transpose(1, 2)
cat_x  = torch.concat((
   trf_x[:, :, None, :].expand(-1, -1, transformed_window, -1), 
   trf_x[:, None, :, :].expand(-1, transformed_window, -1, -1)
), dim=-1)

w = nn.Linear(in_features=2*temporal_patterns, out_features=1, bias=False)
leaky_relu = nn.LeakyReLU(negative_slope=0.01)
alpha = torch.softmax(leaky_relu(w(cat_x)), dim=2)
alpha = alpha.squeeze(-1)

In [11]:
class TimeOrientedGAT(nn.Module): 
    def __init__(self, transformed_window: int, temporal_patterns: int, negative_slope: float = 0.01): 
        super(TimeOrientedGAT, self).__init__()
        self.transformed_window = transformed_window
        self.temporal_patterns = temporal_patterns
        self.negative_slope = negative_slope
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        cat_x  = torch.concat((
            x[:, :, None, :].expand(-1, -1, self.transformed_window, -1), 
            x[:, None, :, :].expand(-1, self.transformed_window, -1, -1)
        ), dim=-1)

        w = nn.Linear(in_features=2*self.temporal_patterns, out_features=1, bias=False)
        leaky_relu = nn.LeakyReLU(negative_slope=self.negative_slope)
        alpha = torch.softmax(leaky_relu(w(cat_x)), dim=2)
        alpha = alpha.squeeze(-1)

        return alpha @ x 

In [12]:
tgat = TimeOrientedGAT(transformed_window=transformed_window, temporal_patterns=temporal_patterns)
tattn = tgat(trf_x)

In [13]:
class FeatureOrientedGAT(nn.Module): 
    def __init__(self, transformed_window: int, temporal_patterns: int, negative_slope: float = 0.01): 
        super(FeatureOrientedGAT, self).__init__()
        self.transformed_window = transformed_window
        self.temporal_patterns = temporal_patterns
        self.negative_slope = negative_slope
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feature_x = x.transpose(1, 2)
        cat_x  = torch.concat((
            feature_x[:, :, None, :].expand(-1, -1, self.temporal_patterns, -1), 
            feature_x[:, None, :, :].expand(-1, self.temporal_patterns, -1, -1)
        ), dim=-1)

        w = nn.Linear(in_features=2*self.transformed_window, out_features=1, bias=False)
        leaky_relu = nn.LeakyReLU(negative_slope=self.negative_slope)
        alpha = torch.softmax(leaky_relu(w(cat_x)), dim=2)
        alpha = alpha.squeeze(-1)

        return (alpha @ feature_x).transpose(1, 2)

fgat = FeatureOrientedGAT(transformed_window=transformed_window, temporal_patterns=temporal_patterns)
fattn = fgat(trf_x)
fattn


tensor([[[-0.0829, -0.0814, -0.0829,  ..., -0.0829, -0.0829, -0.0816],
         [-0.0815, -0.0801, -0.0815,  ..., -0.0815, -0.0815, -0.0803],
         [-0.0658, -0.0645, -0.0658,  ..., -0.0658, -0.0658, -0.0647],
         ...,
         [-0.0696, -0.0683, -0.0696,  ..., -0.0696, -0.0696, -0.0685],
         [-0.0730, -0.0717, -0.0730,  ..., -0.0730, -0.0730, -0.0719],
         [-0.0754, -0.0741, -0.0754,  ..., -0.0754, -0.0754, -0.0743]],

        [[-0.0471, -0.0447, -0.0488,  ..., -0.0477, -0.0488, -0.0445],
         [-0.0479, -0.0458, -0.0496,  ..., -0.0486, -0.0495, -0.0455],
         [-0.0517, -0.0494, -0.0534,  ..., -0.0523, -0.0533, -0.0492],
         ...,
         [-0.0463, -0.0442, -0.0479,  ..., -0.0469, -0.0478, -0.0439],
         [-0.0480, -0.0458, -0.0496,  ..., -0.0486, -0.0495, -0.0456],
         [-0.0474, -0.0454, -0.0490,  ..., -0.0480, -0.0489, -0.0451]],

        [[-0.0327, -0.0318, -0.0331,  ..., -0.0324, -0.0329, -0.0325],
         [-0.0345, -0.0336, -0.0349,  ..., -0

In [14]:
X = torch.concat((
    machine_embeds[:, None, :].expand(-1, transformed_window, -1), 
    tattn, 
    trf_x, 
    fattn, 
), dim=-1)

encoder = nn.GRU(
    input_size=3 * temporal_patterns + 100, 
    hidden_size=temporal_patterns, 
    num_layers=50, 
    batch_first=True
)

output, hn = encoder(X)

class Encoder(nn.Module): 
    def __init__(self, embedding_size: int, temporal_patterns: int, num_layers: int): 
        super(Encoder, self).__init__()
        self.gru = nn.GRU(
            input_size=3*temporal_patterns + embedding_size, 
            hidden_size=temporal_patterns, 
            num_layers=num_layers, 
            batch_first=True
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output, hn = self.gru(x)
        return output 

gru = Encoder(embedding_size=100, temporal_patterns=temporal_patterns, num_layers=50)
output = gru(X)


In [15]:
hn.shape

torch.Size([50, 32, 20])

In [16]:
hn = output[:, -1, :]
hn = hn[:, None, :]

net1 = nn.Linear(in_features=temporal_patterns, out_features=channels)
net2 = nn.Linear(in_features=channels, out_features=channels)
net3 = nn.Linear(in_features=channels, out_features=channels)

y_pred = net3(net2(net1(hn))) 
y_pred



tensor([[[ 0.0745,  0.0628,  0.1437,  ..., -0.0961,  0.1279,  0.0943]],

        [[ 0.0745,  0.0628,  0.1437,  ..., -0.0961,  0.1279,  0.0943]],

        [[ 0.0745,  0.0628,  0.1437,  ..., -0.0961,  0.1279,  0.0943]],

        ...,

        [[ 0.0745,  0.0628,  0.1437,  ..., -0.0961,  0.1279,  0.0943]],

        [[ 0.0745,  0.0628,  0.1437,  ..., -0.0961,  0.1279,  0.0943]],

        [[ 0.0745,  0.0628,  0.1437,  ..., -0.0961,  0.1279,  0.0943]]],
       grad_fn=<ViewBackward0>)

In [17]:
target = labels[:, None, :]

In [18]:
loss = nn.MSELoss()
loss_value = loss(y_pred, target)

In [21]:
hn

tensor([[[ 0.0483, -0.0148,  0.0847, -0.2676, -0.1665, -0.2487, -0.0747,
           0.0944, -0.0242, -0.2879, -0.1058, -0.2689,  0.0449, -0.0538,
          -0.0245,  0.1966,  0.0548,  0.0834,  0.0650, -0.0123]],

        [[ 0.0483, -0.0148,  0.0847, -0.2676, -0.1665, -0.2487, -0.0747,
           0.0944, -0.0242, -0.2879, -0.1058, -0.2689,  0.0449, -0.0538,
          -0.0245,  0.1966,  0.0548,  0.0834,  0.0650, -0.0123]],

        [[ 0.0483, -0.0148,  0.0847, -0.2676, -0.1665, -0.2487, -0.0747,
           0.0944, -0.0242, -0.2879, -0.1058, -0.2689,  0.0449, -0.0538,
          -0.0245,  0.1966,  0.0548,  0.0834,  0.0650, -0.0123]],

        [[ 0.0483, -0.0148,  0.0847, -0.2676, -0.1665, -0.2487, -0.0747,
           0.0944, -0.0242, -0.2879, -0.1058, -0.2689,  0.0449, -0.0538,
          -0.0245,  0.1966,  0.0548,  0.0834,  0.0650, -0.0123]],

        [[ 0.0483, -0.0148,  0.0847, -0.2676, -0.1665, -0.2487, -0.0747,
           0.0944, -0.0242, -0.2879, -0.1058, -0.2689,  0.0449, -0.0538,
  

In [ ]:
latent_dim = 45

logvar = nn.Linear(in_features=temporal_patterns, out_features=latent_dim)
mu = nn.Linear(in_features=temporal_patterns, out_features=latent_dim)

logvariance = logvar(hn)
mean = mu(hn)

z = mean + torch.exp(0.5 * logvariance) * torch.randn_like(logvariance)
z = z.expand(-1, 100, -1)

hidden_size: int = 64
decoder = nn.GRU(input_size=latent_dim, hidden_size=hidden_size, num_layers=4, batch_first=True)
output, ht = decoder(z)

reconstructor = nn.Linear(in_features=hidden_size, out_features=channels)
x = reconstructor(output)

tensor([[[ 0.0037, -0.0278, -0.0513,  ..., -0.0251,  0.0293, -0.0637],
         [ 0.0192, -0.0184, -0.0434,  ..., -0.0145,  0.0325, -0.0777],
         [ 0.0336, -0.0122, -0.0467,  ..., -0.0150,  0.0274, -0.0768],
         ...,
         [ 0.0663, -0.0040, -0.0956,  ..., -0.0538,  0.0072, -0.0273],
         [ 0.0663, -0.0040, -0.0956,  ..., -0.0538,  0.0072, -0.0273],
         [ 0.0663, -0.0040, -0.0956,  ..., -0.0538,  0.0072, -0.0273]],

        [[ 0.0027, -0.0293, -0.0469,  ..., -0.0253,  0.0301, -0.0681],
         [ 0.0150, -0.0225, -0.0287,  ..., -0.0137,  0.0337, -0.0908],
         [ 0.0240, -0.0194, -0.0177,  ..., -0.0114,  0.0282, -0.1004],
         ...,
         [ 0.0102, -0.0136,  0.0154,  ..., -0.0225,  0.0056, -0.0857],
         [ 0.0102, -0.0136,  0.0154,  ..., -0.0225,  0.0056, -0.0857],
         [ 0.0102, -0.0136,  0.0154,  ..., -0.0225,  0.0056, -0.0857]],

        [[-0.0036, -0.0321, -0.0453,  ..., -0.0158,  0.0290, -0.0658],
         [-0.0021, -0.0313, -0.0259,  ...,  0

In [ ]:
z.shape

torch.Size([32, 100, 45])

In [ ]:
class MultiVariateTSGAT(nn.Module): 
    def __init__(
        self, 
        transformed_window, 
        temporal_patterns, 
    ): 
     
        self.transformed_window = transformed_window
        self.temporal_patterns = temporal_patterns
        
        self.feature_attn = FeatureOrientedGAT()
        